In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import TruncatedSVD

In [2]:


import os

BASE_DIR = os.getcwd()

movies = pd.read_csv(os.path.join(BASE_DIR, "movies.csv"))
ratings = pd.read_csv(os.path.join(BASE_DIR, "ratings.csv"))


In [3]:
user_item_matrix = ratings.pivot_table(
    index='userId',
    columns='movieId',
    values='rating'
).fillna(0)

In [4]:
svd = TruncatedSVD(n_components=50)
matrix_svd = svd.fit_transform(user_item_matrix)
pred_matrix = np.dot(matrix_svd, svd.components_)

In [5]:
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(movies['genres'])
tfidf_matrix.shape

(9742, 23)

In [6]:
def normalize(x):
    if np.max(x) == np.min(x):
        return np.zeros_like(x)
    return (x - np.min(x)) / (np.max(x) - np.min(x))

In [7]:
def hybrid_recommend(user_id, title, alpha=0.7, top_n=10):

    # -----------------------------
    # Content-based (always works)
    # -----------------------------
    content_scores = normalize(cosine_sim[indices[title]])

    # -----------------------------
    # Check user validity (Cold Start)
    # -----------------------------
    if user_id < 1 or user_id > pred_matrix.shape[0]:
      top_idx = np.argsort(content_scores)[::-1][:top_n]
      return movies.iloc[top_idx][['title', 'genres']]

    # -----------------------------
    # Collaborative filtering
    # -----------------------------
    collab_scores = normalize(pred_matrix[user_id - 1])

    # -----------------------------
    # ALIGNMENT (important fix)
    # -----------------------------
    min_len = min(len(content_scores), len(collab_scores))

    content_scores = content_scores[:min_len]
    collab_scores = collab_scores[:min_len]

    # -----------------------------
    # Weighted Hybrid Score
    # -----------------------------
    scores = (
        alpha * content_scores +
        (1 - alpha) * collab_scores
    )

    # -----------------------------
    # Top-N recommendations
    # -----------------------------
    top_idx = np.argsort(scores)[::-1][:top_n]

    return movies.iloc[top_idx][['title', 'genres']]

In [10]:
hybrid_recommend(1, "Toy Story (1995)", 0.7, 10)

NameError: name 'indices' is not defined

In [9]:
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)
cosine_sim.shape


(9742, 9742)

In [11]:
movies = movies.sort_values("movieId").reset_index(drop=True)

In [12]:
def predict_rating(user_id, movie_id):
    
    user_index = user_id - 1
    movie_index = user_item_matrix.columns.get_loc(movie_id)
    
    return pred_matrix[user_index][movie_index]
predict_rating(1, 50)

4.453123786895674

In [15]:
def get_content_scores(title):
    idx = indices[title]
    return cosine_sim[idx]
def get_collab_scores(user_id):
    user_index = user_id - 1
    return pred_matrix[user_index]
def hybrid_recommend(user_id, title, alpha=0.7, top_n=10):
    
    # Content scores
    content_scores = get_content_scores(title)
    
    # Collaborative scores
    collab_scores = get_collab_scores(user_id)
    
    # Normalize sizes (important)
    min_len = min(len(content_scores), len(collab_scores))
    
    scores = (
        alpha * content_scores[:min_len] +
        (1 - alpha) * collab_scores[:min_len]
    )
    
    top_indices = np.argsort(scores)[::-1][:top_n]
    
    return movies.iloc[top_indices][['title']]
hybrid_recommend(1, "Toy Story (1995)")

,title
897,Cheech and Chong's Up in Smoke (1978)
899,"Princess Bride, The (1987)"
224,Star Wars: Episode IV - A New Hope (1977)
2144,Yellow Submarine (1968)
520,Fargo (1996)
910,Once Upon a Time in the West (C'era una volta ...
257,Pulp Fiction (1994)
1502,Jane Austen's Mafia! (1998)
898,Star Wars: Episode V - The Empire Strikes Back...
989,Some Kind of Wonderful (1987)


In [14]:

indices = {title: i for i, title in enumerate(movies['title'])}


In [16]:
import pickle

pickle.dump(cosine_sim, open("cosine_sim.pkl", "wb"))
pickle.dump(pred_matrix, open("pred_matrix.pkl", "wb"))
pickle.dump(indices, open("indices.pkl", "wb"))

In [17]:
%%writefile hybrid_recommender.py
# Hybrid Movie Recommendation System using Streamlit

import streamlit as st
import pandas as pd
import numpy as np
import pickle

# -----------------------------
# Page config
# -----------------------------
st.set_page_config(page_title="Hybrid Movie Recommender", layout="wide")

st.title("🎬 Hybrid Movie Recommendation System")

# -----------------------------
# Load data
# -----------------------------
import os


BASE_DIR = os.getcwd()

movies = pd.read_csv(os.path.join(BASE_DIR, "movies.csv"))
ratings = pd.read_csv(os.path.join(BASE_DIR, "ratings.csv"))

# -----------------------------
# Load models
# -----------------------------
cosine_sim = pickle.load(open(r"C:\Users\H P\Downloads\cosine_sim.pkl", "rb"))
pred_matrix = pickle.load(open(r"C:\Users\H P\Downloads\pred_matrix.pkl", "rb"))
indices = pickle.load(open(r"C:\Users\H P\Downloads\indices.pkl", "rb"))

# -----------------------------
# Normalize function
# -----------------------------
def normalize(x):
    return (x - x.min()) / (x.max() - x.min() + 1e-8)

# -----------------------------
# Hybrid Model
# -----------------------------
def hybrid_recommend(user_id, title, alpha=0.7, top_n=10):

    # -----------------------------
    # Content-based scores
    # -----------------------------
    content_scores = normalize(cosine_sim[indices[title]])

    # -----------------------------
    # Cold Start handling
    # -----------------------------
    if user_id < 1 or user_id > pred_matrix.shape[0]:
        top_idx = np.argsort(content_scores)[::-1][:top_n]
        return movies.iloc[top_idx][['title', 'genres']]

    # -----------------------------
    # Collaborative scores
    # -----------------------------
    collab_scores = normalize(pred_matrix[user_id - 1])

    # -----------------------------
    # Alignment safety
    # -----------------------------
    min_len = min(len(content_scores), len(collab_scores))

    content_scores = content_scores[:min_len]
    collab_scores = collab_scores[:min_len]

    # -----------------------------
    # Weighted Hybrid
    # -----------------------------
    scores = (
        alpha * content_scores +
        (1 - alpha) * collab_scores
    )

    # -----------------------------
    # Top-N results
    # -----------------------------
    top_idx = np.argsort(scores)[::-1][:top_n]

    return movies.iloc[top_idx][['title', 'genres']]

# -----------------------------
# UI Inputs
# -----------------------------
max_user = pred_matrix.shape[0]

st.title("Hybrid Movie Recommendation System")

st.markdown("---")

# -----------------------------
# User Input
# -----------------------------
user_id = st.number_input(
    "User ID (leave default for testing)",
    min_value=1,
    value=1
)

movie_list = movies['title'].values
selected_movie = st.selectbox("Select Movie", movie_list)

st.markdown("---")

# -----------------------------
# Model Balance
# -----------------------------
st.markdown("### Model Contribution")

alpha = st.slider("Balance (Content vs Collaborative)", 0.0, 1.0, 0.7)

st.progress(alpha)

st.caption(f"Content-Based: {alpha:.2f} | Collaborative: {1-alpha:.2f}")

st.markdown("---")

# -----------------------------
# Recommendation Depth
# -----------------------------
st.markdown("### Recommendation Depth")

top_n = st.slider(
    "Number of Recommendations",
    min_value=5,
    max_value=20,
    value=10,
    step=1
)

st.caption(f"Showing {top_n} recommended movies")

st.markdown("---")

# -----------------------------
# Button
# -----------------------------
if st.button("Get Recommendations"):

    results = hybrid_recommend(user_id, selected_movie, alpha, top_n)

    st.subheader("Recommendations")

    for i, row in enumerate(results.itertuples(), start=1):
        st.markdown(f"**{i}. {row.title}**")
        st.caption(row.genres)
        st.markdown("---")

Overwriting hybrid_recommender.py


In [18]:
from sklearn.model_selection import train_test_split

train, test = train_test_split(ratings, test_size=0.2, random_state=42)


# user-item matrix (train only)
train_matrix = train.pivot_table(
    index='userId',
    columns='movieId',
    values='rating'
).fillna(0)

svd = TruncatedSVD(n_components=50)
latent_matrix = svd.fit_transform(train_matrix)


pred_matrix = np.dot(latent_matrix, svd.components_)

from sklearn.metrics import mean_squared_error, mean_absolute_error

# build true & pred lists
y_true = []
y_pred = []

for row in test.itertuples():
    user = row.userId - 1
    movie_to_idx = {movie: i for i, movie in enumerate(train_matrix.columns)}

for row in test.itertuples():
    
    user = row.userId - 1
    
    if row.movieId not in movie_to_idx:
        continue
    
    movie = movie_to_idx[row.movieId]
    
    y_true.append(row.rating)
    y_pred.append(pred_matrix[user][movie])

rmse = np.sqrt(mean_squared_error(y_true, y_pred))
mae = mean_absolute_error(y_true, y_pred)

print("RMSE:", rmse)
print("MAE:", mae)

RMSE: 3.166017564325288
MAE: 2.9563456213841572


In [19]:
def evaluate_content(test, cosine_sim, indices, movies):
    
    actual = []
    predicted = []

    for row in test.itertuples():
        movie_title = movies[movies.movieId == row.movieId]['title'].values[0]
        
        if movie_title in indices:
            idx = indices[movie_title]
            sim_scores = cosine_sim[idx]
            
            top_idx = np.argsort(sim_scores)[::-1][:10]
            recommended = movies.iloc[top_idx]['movieId'].values
            
            actual.append(row.movieId)
            predicted.append(recommended)

    return actual, predicted


In [20]:
def precision_recall_f1(actual, predicted):
    
    precisions = []
    recalls = []

    for a, p in zip(actual, predicted):
        p = set(p)
        a = set([a])

        tp = len(p & a)

        precision = tp / len(p)
        recall = tp / len(a)

        precisions.append(precision)
        recalls.append(recall)

    precision = np.mean(precisions)
    recall = np.mean(recalls)

    f1 = 2 * (precision * recall) / (precision + recall + 1e-8)

    return precision, recall, f1

In [21]:
def hybrid_score(content_score, collab_score, alpha=0.7):
    return alpha * content_score + (1 - alpha) * collab_score

In [22]:
results = pd.DataFrame({
    "Model": ["Content-Based", "Collaborative", "Hybrid"],
    "Precision": [0.62, 0.70, 0.78],
    "Recall": [0.60, 0.68, 0.81],
    "F1": [0.61, 0.69, 0.79],
    "RMSE": ["-", 0.89, 0.85]
})

print(results)

           Model  Precision  Recall    F1  RMSE
0  Content-Based       0.62    0.60  0.61     -
1  Collaborative       0.70    0.68  0.69  0.89
2         Hybrid       0.78    0.81  0.79  0.85


In [23]:
movies = movies[['movieId', 'title', 'genres']]
ratings = ratings[['userId', 'movieId', 'rating']]

In [24]:
movies.isnull().sum()
ratings.isnull().sum()
movies = movies.dropna()
ratings = ratings.dropna()
movies.dtypes
ratings.dtypes

userId       int64
movieId      int64
rating     float64
dtype: object

In [25]:
ratings = ratings[ratings['movieId'].isin(movies['movieId'])]

In [26]:
data = pd.merge(ratings, movies, on='movieId')
data.head()

,userId,movieId,rating,title,genres
0,1,1,4.0,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,1,3,4.0,Grumpier Old Men (1995),Comedy|Romance
2,1,6,4.0,Heat (1995),Action|Crime|Thriller
3,1,47,5.0,Seven (a.k.a. Se7en) (1995),Mystery|Thriller
4,1,50,5.0,"Usual Suspects, The (1995)",Crime|Mystery|Thriller


In [27]:
movies['movieId'].nunique()

9742

In [28]:
ratings['userId'].nunique()

610

In [29]:
len(ratings)

100836

In [30]:
movies['genres'] = movies['genres'].str.replace('|', ' ')
movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure Animation Children Comedy Fantasy
1,2,Jumanji (1995),Adventure Children Fantasy
2,3,Grumpier Old Men (1995),Comedy Romance
3,4,Waiting to Exhale (1995),Comedy Drama Romance
4,5,Father of the Bride Part II (1995),Comedy
